# Functors

In this first part of our series on monads, we aren't going to talk about monads at all! Instead we will talk about functors. Monads are a somewhat tricky idea. So to understand them better, it helps to start with functors, which are simpler in every way. If you already understand functors, you can move on to part 2 to learn about applicative functors.

## What is a Functor?

A Functor is any type that can act as a generic container. A Functor allows us to transform the underlying values with a function, so that the values are all updated, but the structure of the container is the same. This is called `"mapping"`.

## How are Functors represented in Haskell?

Haskell represents the concept of a functor with the **Functor** typeclass. This typeclass has a single required function **fmap**.

```
class Functor f where
  fmap :: (a -> b) -> f a -> f b
```
We can see that **fmap** is a higher ordered function taking two inputs.

-  A transformation function from an a type to a b type (a and b can be the same if we want)
-  A functor containing values of type a

The output it produces is a new functor containing values of type b. This new functor has exactly the same structure (or shape) as the input functors; all that has changed is that each element has been modified by the input function.

## Basic Functor Examples

A List is the most basic example of a functor. A list contains 0 or more elements of the same, underlying type. When we map a function over a list, we create a new list with the same number of elements where each item has been modified.

Let's see how this works in code.

In [3]:
let list1 = [] :: [String]
let list2 = [1, 2, 3] :: [Int]
let list3 = [True, False, True, False] :: [Bool]

-- For each of the underlying types above,
-- we can think of a transformation function.

let f1 = length :: String -> Int
let f2 = (* 2) :: Int -> Int
let f3 = show :: Bool -> String

We can use fmap to apply each function to all elements of the input list. And the resulting list has the same "structure" (length), with each element corresponding to an element in the original list.

In [4]:
fmap f1 list1
:t fmap f1 list1

[]

fmap f1 list1 :: [Int]

In [5]:
-- infix fmap

f2 <$> list2
:t f2 <$> list2

[2,4,6]

f2 <$> list2 :: [Int]

In [6]:
-- another prefixed option

(<$>) f3 list3
:t (<$>) f3 list3

["True","False","True","False"]

(<$>) f3 list3 :: [String]

Another simple example of a functor is the **Maybe** type. This object can contain a value of a particular type as **Just**, or it is **Nothing** (like a null value).

In [7]:
let maybe1 = Just "Hello" :: Maybe String
let maybe2 = Just 6 :: Maybe Int
let maybe3 = Nothing :: Maybe Bool

When we apply fmap, the "structure" of the object stays the same. With a list, this meant the size. With a Maybe, it means that if our input is Nothing, the output will be Nothing, and likewise with Just. Let's try using fmap with the same functions from above.

In [8]:
fmap f1 maybe1
fmap f2 maybe2
fmap f3 maybe3

:t fmap f1 maybe1
:t fmap f2 maybe2
:t fmap f3 maybe3

Just 5

Just 12

Nothing

fmap f1 maybe1 :: Maybe Int

fmap f2 maybe2 :: Maybe Int

fmap f3 maybe3 :: Maybe String

## How do I make a Functor instance?

You can make a Functor instance for any parameterized type that can act as a container. The only thing you really need to fill in is the fmap implementation. It will just look like this:

```
instance Functor MyType where
  fmap f (MyType ...) = ...
```

For a quick example, let's suppose we have a type describing the measurements of a home or apartment. We might want these measurements in different types or units, so we'll use a type parameter.

In [9]:
data Measurements a = Measurements
  { totalSize :: a
  , numBedrooms :: Int
  , masterBedroomSize :: a
  , livingRoomSize :: a
  } deriving (Show, Eq)

Now suppose we have a function to convert the measurement type. Here's all we need to do to implement the functor instance.

In [10]:
instance Functor Measurements where
  fmap convertMeasure (Measurements ts nb mbs lrs) = Measurements
    (convertMeasure ts) nb (convertMeasure mbs) (convertMeasure lrs)

Now we can define a number of conversion functions, for possible types we might store inside these measurements. We can have some newtype wrappers to encapsulate the units as well.

In [11]:
newtype SquareFeet = SquareFeet Double
  deriving (Show, Eq)

newtype SquareMeters = SquareMeters Double
  deriving (Show, Eq)

squareFeetToMeters :: SquareFeet -> SquareMeters
squareFeetToMeters (SquareFeet ft) = SquareMeters (ft / 10.764)

And now let's see our functor in action!

In [ ]:
let m1 = Measurements 1200 3 200 700 :: Measurements Int
let m2 = fmap fromIntegral m1 :: Measurements Double
let m3 = fmap SquareFeet m2
let m4 = squareFeetToMeters <$> m3


In [13]:
m1

Measurements {totalSize = 1200, numBedrooms = 3, masterBedroomSize = 200, livingRoomSize = 700}

In [14]:
m2

Measurements {totalSize = 1200.0, numBedrooms = 3, masterBedroomSize = 200.0, livingRoomSize = 700.0}

In [15]:
m3
:t m3

Measurements {totalSize = SquareFeet 1200.0, numBedrooms = 3, masterBedroomSize = SquareFeet 200.0, livingRoomSize = SquareFeet 700.0}

m3 :: Measurements SquareFeet

In [16]:
m4

Measurements {totalSize = SquareMeters 111.48272017837236, numBedrooms = 3, masterBedroomSize = SquareMeters 18.58045336306206, livingRoomSize = SquareMeters 65.03158677071721}

First we convert our integers to floating point values with fromIntegral, then we wrap them in SquareFeet, and finally convert to SquareMeters. In the last example, we use the operator (<$>) which is equivalent tofmap!

```
(<$>) :: (Functor f) => (a -> b) -> f a -> f b
(<$>) = fmap

```

## What are the Functor Laws?

Functors have two laws:

-  identity 
-  composition

 These laws express behaviors that your functor instances should follow. If they don't, other programmers will be very confused at the effect your instances have on the program. Many structures have similar laws, including monads.

The `identity law` says that if you `"map" the identity function (id) over your functor`, the resulting functor should be the same. A succinct way of stating this is:

```
fmap id = id

```

The identity function shouldn't change underlying elements, so it shouldn't change the functor either!

-----

The `composition law` says that if we `"map" two functions in succession over our functor`, this should be the same as "composing" the functions and simply mapping that one super-function.

```
(fmap f) . (fmap g) = fmap (f . g)
```

As an example, we can make a super-function of converting a double into SquareMeters by going through SquareFeet first:

In [21]:
-- FIX: The function composition order was reversed in the article.
-- In Haskell, (f . g) means g runs first, then f takes the result (i.e., f(g(x))).
-- Correct order: squareFeetToMeters . SquareFeet

doubleToSquareMeters :: Double -> SquareMeters
doubleToSquareMeters = squareFeetToMeters . SquareFeet

We can apply the functions separately (m1 -> m2 -> m3), or we can apply the super-function once (m4). The results (m3 and m4) are the same!



In [19]:
let m1 = Measurements 1200.0 3 200.0 700.0 :: Measurements Double
let m2 = fmap SquareFeet m1 :: Measurements SquareFeet
let m3 = fmap squareFeetToMeters m2
let m4 = fmap doubleToSquareMeters m1
m3 == m4

True

If this is confusing, don't worry. If you make your instances intuitively, they should almost certainly follow the laws! To break them, you would have to do something like this, introducing an arbitrary value into the instance:

In [22]:
fmap convertMeasure (Measurements ts nb mbs lrs) = Measurements
    0 nb (convertMeasure mbs) (convertMeasure lrs)

Now if we map the id function, we'll always get 0 for the total size, even if the input's total size is non-zero.

##  How do Functors Help with Monads?


Functors are an abstract mathematical structure that we represent in Haskell with a typeclass, and this structure has particular "laws" associated with it that dictate its expected behavior. Monads are the same way!

However, functors are simpler to understand. The functor typeclass has only one function and two straightforward laws. We can easily visualize what they do. Monads meanwhile have multiple functions and several more complicated laws.

Understanding abstract mathematical structures is a little tricky for most people. So it helps to start with a simpler idea like functors before we try to understand monads.

## Conclusion

Hopefully this first part of our series has given you a better understanding of functors. To continue your journey of understanding monads and other functional structures, you can move on to part 2 where we'll talk about Applicative Functors!